# Part 1: Mainframe operator scheduling (Linear Programming)
Weekly scheduling of 6 student operators with PuLP. No external data is needed.

1. **Baseline**: minimise labour cost subject to coverage, availability and minimum hours
2. **Fairness scenarios**: minimise deviation from the average workload, with and without a +1.8% budget cap
3. **Skill-based**: at least 6 h/day of Programming and 6 h/day of Troubleshooting skills

## 1. Baseline: cost minimisation

In [1]:
import pulp
import pandas as pd


# ==============================================================================
# 1. GLOBAL DATA PREPARATION
# ==============================================================================

# Operators and Time
operators = ['E. Khan', 'Y. Chen', 'A. Taylor', 'R. Zidane', 'R. Perez', 'C. Santos']
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri']
n_days = len(days)

# Wages (£/hr)
wages = {
    'E. Khan': 25, 'Y. Chen': 26, 'A. Taylor': 24, 
    'R. Zidane': 23, 'R. Perez': 28, 'C. Santos': 30
}

# Student Types & Minimum Weekly Hours
bachelors = ['E. Khan', 'Y. Chen', 'A. Taylor', 'R. Zidane'] # Min 8 hours
masters = ['R. Perez', 'C. Santos']                         # Min 7 hours

min_weekly_hours = {name: 8 for name in bachelors}
min_weekly_hours.update({name: 7 for name in masters})

# Availability Matrix (Max hours per day)
availability = {
    "E. Khan":  {"Mon": 6, "Tue": 0, "Wed": 6, "Thu": 0, "Fri": 6},
    "Y. Chen":  {"Mon": 0, "Tue": 6, "Wed": 0, "Thu": 6, "Fri": 0},
    "A. Taylor": {"Mon": 4, "Tue": 8, "Wed": 4, "Thu": 0, "Fri": 4},
    "R. Zidane": {"Mon": 5, "Tue": 5, "Wed": 5, "Thu": 0, "Fri": 5},
    "R. Perez":  {"Mon": 3, "Tue": 0, "Wed": 3, "Thu": 8, "Fri": 0},
    "C. Santos": {"Mon": 0, "Tue": 0, "Wed": 0, "Thu": 6, "Fri": 2}
}
# Skill Sets (For Task 3)
prog_skills = ['E. Khan', 'Y. Chen', 'R. Perez', 'C. Santos']
trouble_skills = ['A. Taylor', 'R. Zidane', 'C. Santos']

# General Constraints
daily_required_hours = 14
avg_target_hours = 70 / len(operators)  # For Task 2 Fairness (11.67 hrs)


# ==============================================================================
# 2. TASK 1: Minimize Total Labour Cost
# ==============================================================================

# Initialize Problem
prob1 = pulp.LpProblem("Task1_Minimize_Cost", pulp.LpMinimize)

# Decision Variables: Hours per operator per day
x1 = pulp.LpVariable.dicts("t1_hrs", (operators, range(n_days)), lowBound=0)

# --- Constraints ---

# 1. Daily Coverage: Total hours must be exactly 14 per day
for d in range(n_days):
    prob1 += pulp.lpSum([x1[op][d] for op in operators]) == daily_required_hours

# 2. Individual Availability: Hours cannot exceed operator's max daily availability
for op in operators:
    for d in range(n_days):
        prob1 += x1[op][d] <= availability[op][days[d]]

# 3. Weekly Minimum Commitment: Bachelors >= 8, Masters >= 7
for op in operators:
    prob1 += pulp.lpSum([x1[op][d] for d in range(n_days)]) >= min_weekly_hours[op]

# --- Objective Function ---
# Minimize Total Wage Cost
prob1 += pulp.lpSum([x1[op][d] * wages[op] for op in operators for d in range(n_days)])

# Solve
prob1.solve(pulp.PULP_CBC_CMD(msg=0))
task1_cost = pulp.value(prob1.objective)

# result
print(f"Status: {pulp.LpStatus[prob1.status]}")

if pulp.LpStatus[prob1.status] == 'Optimal':
    # 1. Prepare data dictionary
    results_data = []
    
    for op in operators:
        row = {'Operator': op} # First column is the name
        total_hours = 0
        
        # Fill in daily working hours
        for d in range(5):
            hours = pulp.value(x1[op][d])
            row[days[d]] = hours
            total_hours += hours
            
        # Calculate total hours and wages
        row['Total Hours'] = total_hours
        row['Hour wage'] = wages[op]
        row['Weekly Cost'] = total_hours * wages[op]
        
        results_data.append(row)

    # 2. Convert to DataFrame
    df_schedule = pd.DataFrame(results_data)
    
    # Set Operator as index (for better formatting)
    df_schedule.set_index('Operator', inplace=True)

    # 3. Display results
    print("\n=== Weekly Staff Schedule (Task 1) ===")
    # pd.set_option('display.max_columns', None) # Ensure all columns are displayed
    # pd.set_option('display.width', 1000)
    print(df_schedule)
    
    print("-" * 60)
    
    # 4. Display total cost (for verification)
    total_labour_cost = pulp.value(prob1.objective)
    print(f"Total Labour Cost: £{total_labour_cost:.2f}")

else:
    print("No optimal solution found.")

Status: Optimal

=== Weekly Staff Schedule (Task 1) ===
           Mon  Tue  Wed  Thu  Fri  Total Hours  Hour wage  Weekly Cost
Operator                                                               
E. Khan    2.0  0.0  4.0  0.0  3.0          9.0         25        225.0
Y. Chen    0.0  2.0  0.0  6.0  0.0          8.0         26        208.0
A. Taylor  4.0  7.0  4.0  0.0  4.0         19.0         24        456.0
R. Zidane  5.0  5.0  5.0  0.0  5.0         20.0         23        460.0
R. Perez   3.0  0.0  1.0  3.0  0.0          7.0         28        196.0
C. Santos  0.0  0.0  0.0  5.0  2.0          7.0         30        210.0
------------------------------------------------------------
Total Labour Cost: £1755.00


## 2. Cost vs fairness scenarios

In [2]:
import pulp

# --- 1. Data Preparation ---
task1_cost = 1755  
cost_multiplier = 1.018  # Scenario i: 1.8% increase limit
max_budget_i = task1_cost * cost_multiplier #calculate the budget cap for scenario i

daily_required_hours = 14 #hours required to be covered per day
avg_target = 70 / len(operators)  # average target hours: 11.67 hours/week

# ==========================================================
# SCENARIO I: Fairer distribution (Budget limited to +1.8%)
# ==========================================================
#initialize problem object, objective is to minimize total deviation
prob_i = pulp.LpProblem("Scenario_i_Limited_Budget", pulp.LpMinimize)

#define decision variables:
x_i = pulp.LpVariable.dicts("h_i", (operators, days), lowBound=0, cat="Integer")
#dev_i: deviation of hours for each operator (must be >=0)
dev_i = pulp.LpVariable.dicts("dev_i", operators, lowBound=0)

#---set objective function---
#objective: minimize the sum of deviations for all operators
prob_i += pulp.lpSum([dev_i[o] for o in operators])

#---set constraints ---
#1. budget constraints
#calculate current total wage cost
cost_i = pulp.lpSum([x_i[o][d] * wages[o] for o in operators for d in days])
#limit total cost to not exceed max_budget_i
prob_i += cost_i <= max_budget_i, "Budget_Constraint_i"

# 2. Individual Operator Constraints
for o in operators:
    # Calculate total weekly hours for the operator
    total_h = pulp.lpSum([x_i[o][d] for d in days])
    # Fairness Deviation Constraints (Linearization of absolute value)
    # We want dev >= |total_h - avg_target|
    # So we split this into two inequalities:
    # (1) dev >= Actual - Target
    prob_i += dev_i[o] >= total_h - avg_target
    # (2) dev >= Target - Actual
    prob_i += dev_i[o] >= avg_target - total_h
    # Minimum weekly hours constraint (Bachelors >= 8, Masters >= 7)
    prob_i += total_h >= min_weekly_hours[o]
    # Daily availability constraint (cannot exceed operator's available hours for that day)
    for d in days:
        prob_i += x_i[o][d] <= availability[o][d]

# 3. Daily Coverage Constraint
# Total hours across all operators must equal exactly 14 hours per day
for d in days:
    prob_i += pulp.lpSum([x_i[o][d] for o in operators]) == daily_required_hours

# Solve the model
prob_i.solve(pulp.PULP_CBC_CMD(msg=0))

# ==========================================================
# SCENARIO II: Fairest possible (No budget limit)
# ==========================================================
# Initialize second problem object
prob_ii = pulp.LpProblem("Scenario_ii_Unlimited_Budget", pulp.LpMinimize)

# Define new variables (using _ii suffix to avoid confusion with Scenario I)
x_ii = pulp.LpVariable.dicts("h_ii", (operators, days), lowBound=0, cat="Integer")
dev_ii = pulp.LpVariable.dicts("dev_ii", operators, lowBound=0)

# Objective Function: Same as before, minimize total deviation
prob_ii += pulp.lpSum([dev_ii[o] for o in operators])

# --- Set Constraints (Note: No budget constraint here) ---
for o in operators:
    total_h = pulp.lpSum([x_ii[o][d] for d in days])
    # Fairness Deviation Constraints (Same as above)
    prob_ii += dev_ii[o] >= total_h - avg_target
    prob_ii += dev_ii[o] >= avg_target - total_h
    # Minimum weekly hours constraint
    prob_ii += total_h >= min_weekly_hours[o]
    # Daily availability constraint
    for d in days:
        prob_ii += x_ii[o][d] <= availability[o][d]

# Daily Coverage Constraint
for d in days:
    prob_ii += pulp.lpSum([x_ii[o][d] for o in operators]) == daily_required_hours

# Solve the model
prob_ii.solve(pulp.PULP_CBC_CMD(msg=0))

# --- Enhanced Output Results ---
def print_detailed_results(prob, x, title, limit):
    print(f"\n{'='*30}")
    print(f" {title}")
    print(f"{'='*30}")
    
    # Check if an optimal solution was found
    if pulp.LpStatus[prob.status] == 'Optimal':
        # Calculate actual total cost for this scenario
        total_cost = sum(pulp.value(x[o][d]) * wages[o] for o in operators for d in days)
        print(f"Budget Limit: {limit}")
        print(f"Actual Total Cost: £{total_cost:.2f}")
        # Show cost increase compared to Task 1
        print(f"Cost Increase: £{total_cost - task1_cost:.2f} ({((total_cost-task1_cost)/task1_cost)*100:.2f}%)")
        # Show Total Deviation (Objective Value)
        print(f"Total Fairness Deviation: {pulp.value(prob.objective):.2f}\n")
        
        # Summary Table
        print(f"{'Operator':15} | {'Weekly Hours':15} | {'Deviation':10}")
        print("-" * 45)
        for o in operators:
            # Calculate total weekly hours for the operator
            hrs = sum(pulp.value(x[o][d]) for d in days)
            # Print operator's total hours and deviation (absolute value)
            print(f"{o:15} | {hrs:12.0f} | {abs(hrs-avg_target):12.2f}")
        
        # Daily Schedule Table
        print(f"\n--- Daily Work Schedule (Hours) ---")
        # Create header (Operator | Mon | Tue ...)
        header = f"{'Operator':15} | " + " | ".join([f"{d:4}" for d in days])
        print(header)
        print("-" * len(header))
        for o in operators:
            # Get hours for the operator for each day
            row = [f"{int(pulp.value(x[o][d])):4d}" for d in days]
            # Print the full weekly schedule for the operator
            print(f"{o:15} | " + " | ".join(row))
    else:
        print("No Optimal Solution Found.")
        
# Call function to print results for both scenarios
print_detailed_results(prob_i, x_i, "SCENARIO I: FAIRER (+1.8% BUDGET)", f"£{max_budget_i:.2f}")
print_detailed_results(prob_ii, x_ii, "SCENARIO II: FAIREST (UNLIMITED)", "No Limit")


 SCENARIO I: FAIRER (+1.8% BUDGET)
Budget Limit: £1786.59
Actual Total Cost: £1786.00
Cost Increase: £31.00 (1.77%)
Total Fairness Deviation: 14.00

Operator        | Weekly Hours    | Deviation 
---------------------------------------------
E. Khan         |           15 |         3.33
Y. Chen         |           10 |         1.67
A. Taylor       |           12 |         0.33
R. Zidane       |           15 |         3.33
R. Perez        |           11 |         0.67
C. Santos       |            7 |         4.67

--- Daily Work Schedule (Hours) ---
Operator        | Mon  | Tue  | Wed  | Thu  | Fri 
--------------------------------------------------
E. Khan         |    6 |    0 |    5 |    0 |    4
Y. Chen         |    0 |    6 |    0 |    4 |    0
A. Taylor       |    4 |    3 |    1 |    0 |    4
R. Zidane       |    1 |    5 |    5 |    0 |    4
R. Perez        |    3 |    0 |    3 |    5 |    0
C. Santos       |    0 |    0 |    0 |    5 |    2

 SCENARIO II: FAIREST (UNLIMITED)
B

## 3. Skill-based assignment

In [3]:
# ==============================================================================
# 4. TASK 3: Skill-Based Scheduling
# ==============================================================================

# Initialize Problem
prob3 = pulp.LpProblem("Task3_Skills_Model", pulp.LpMinimize)

# Decision Variables
x3 = pulp.LpVariable.dicts("t3_hrs", (operators, range(n_days)), lowBound=0)

# --- Constraints ---

# 1. Daily Coverage: Total hours must be exactly 14 per day
for d in range(n_days):
    prob3 += pulp.lpSum([x3[op][d] for op in operators]) == daily_required_hours

# 2. Individual Availability: Hours cannot exceed availability
for op in operators:
    for d in range(n_days):
        prob3 += x3[op][d] <= availability[op][days[d]]

# 3. Weekly Minimum Commitment
for op in operators:
    prob3 += pulp.lpSum([x3[op][d] for d in range(n_days)]) >= min_weekly_hours[op]

# 4. Skill Constraints (NEW): At least 6 hours of each skill per day
for d in range(n_days):
    # Programming Skills Coverage
    prob3 += pulp.lpSum([x3[op][d] for op in prog_skills]) >= 6
    # Troubleshooting Skills Coverage
    prob3 += pulp.lpSum([x3[op][d] for op in trouble_skills]) >= 6

# --- Objective Function ---
# Minimize Total Wage Cost
prob3 += pulp.lpSum([x3[op][d] * wages[op] for op in operators for d in range(n_days)])

# Solve
prob3.solve(pulp.PULP_CBC_CMD(msg=0))

print(f"Status: {pulp.LpStatus[prob3.status]}")

if pulp.LpStatus[prob3.status] == 'Optimal':
    # 1. Prepare data dictionary
    results_data = []
    
    for op in operators:
        row = {'Operator': op} # First column is the name
        total_hours = 0
        
        # Fill in daily working hours
        for d in range(n_days):
            hours = pulp.value(x3[op][d])
            row[days[d]] = hours
            total_hours += hours
            
        # Calculate total hours and wages
        row['Total Hours'] = total_hours
        row['Hourly Rate'] = wages[op]
        row['Weekly Cost'] = total_hours * wages[op]
        
        results_data.append(row)

    # 2. Convert to DataFrame
    df_schedule = pd.DataFrame(results_data)
    
    # Set Operator as index (for better formatting)
    df_schedule.set_index('Operator', inplace=True)

    # 3. Display results
    print("\n=== Weekly Staff Schedule (Task 3) ===")
    print(df_schedule)
    
    print("-" * 60)
    
    # 4. Display total cost
    total_labour_cost = pulp.value(prob3.objective)
    print(f"Total Labour Cost: £{total_labour_cost:.2f}")
else:
    print("No optimal solution found.")

Status: Optimal

=== Weekly Staff Schedule (Task 3) ===
           Mon  Tue  Wed  Thu  Fri  Total Hours  Hourly Rate  Weekly Cost
Operator                                                                 
E. Khan    5.0  0.0  6.0  0.0  5.0         16.0           25        400.0
Y. Chen    0.0  6.0  0.0  2.0  0.0          8.0           26        208.0
A. Taylor  3.0  3.0  3.0  0.0  3.0         12.0           24        288.0
R. Zidane  5.0  5.0  5.0  0.0  5.0         20.0           23        460.0
R. Perez   1.0  0.0  0.0  6.0  0.0          7.0           28        196.0
C. Santos  0.0  0.0  0.0  6.0  1.0          7.0           30        210.0
------------------------------------------------------------
Total Labour Cost: £1762.00
